# Module 4 - Class 4: SVM vs KNN Showdown
**Khamidullokhon Abduvokhidov**

In [ ]:
# Load Telco data, encode categories, split, and scale features.
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
y = df['Churn'].map({'No': 0, 'Yes': 1})
X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# Train the RBF SVM and record its scores and training time.
start = time.time()
svm = SVC(kernel='rbf', random_state=42).fit(X_train_s, y_train)
svm_time = time.time() - start
y_pred_svm = svm.predict(X_test_s)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
print(f'SVM Accuracy: {svm_accuracy:.4f}, F1: {svm_f1:.4f}, Training Time: {svm_time:.2f}s')

In [ ]:
# Compare KNN models at K values of 3, 5, and 10.
rows = []
for k in [3, 5, 10]:
    start = time.time()
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    elapsed = time.time() - start
    pred = knn.predict(X_test_s)
    rows.append([k, accuracy_score(y_test, pred), f1_score(y_test, pred), elapsed])
knn_results = pd.DataFrame(rows, columns=['K Value', 'Accuracy', 'F1', 'Training Time'])
display(knn_results)
best = knn_results.loc[knn_results['F1'].idxmax()]
best_k = int(best['K Value'])
summary = pd.DataFrame([['SVM (RBF)', svm_accuracy, svm_f1, svm_time], [f'KNN (K={best_k})', best['Accuracy'], best['F1'], best['Training Time']]], columns=['Model', 'Accuracy', 'F1', 'Training Time'])
display(summary)

## Discussion
KNN is useful when local similarity is easy to explain and the data set is small enough for prediction-time distance calculations. SVM is usually preferable in high-dimensional feature spaces and is faster at prediction after training. Both require scaling, while KNN is especially affected by the curse of dimensionality.